# Steps 1-2: inventory + schema cards (Colab-first)
All installs are in the first cell. Put the repo's `qa/` folder and the challenge `data/` folder (CSVs + the source zip) in Google Drive under `municipal_qa/`.

In [ ]:
# 0 - ALL INSTALLS (run once per runtime)
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install pandas openpyxl geopandas pyogrio shapely pyproj rasterio pypdf pdfplumber laspy lxml beautifulsoup4 networkx ifcopenshell pytesseract pillow

In [ ]:
# 1 - paths, Drive, extract sources (idempotent)
import sys, zipfile, os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/municipal_qa')      # contains qa/ and data/
DATA = ROOT/'data'; SRC = Path('/content/sources_extracted'); RUNS = ROOT/'runs'; RUNS.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))
if not (SRC/'sources').is_dir():
    z = next(p for p in DATA.iterdir() if p.stat().st_size > 50_000_000 and zipfile.is_zipfile(p))
    zipfile.ZipFile(z).extractall(SRC)
print(len(os.listdir(SRC/'sources')), 'source files')

In [ ]:
# 2 - Step 1 inventory
from qa.inventory import build_inventory, summarize
inv = build_inventory(DATA, SRC/'sources')
inv.to_csv(RUNS/'inventory.csv', index=False)
print(summarize(inv))

In [ ]:
# 3 - Step 2 schema cards (one per Case_ID) -> runs/cards.json
from qa.cards import build_cards, card_text
import json
packs = build_cards(DATA, SRC/'sources')
(RUNS/'cards.json').write_text(json.dumps(packs, indent=1, ensure_ascii=False), encoding='utf-8')
print(len(packs), 'cards; failed:', [k for k,v in packs.items() if any('card failed' in f for f in v['files'])])